In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 13.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://exterior-everyone-gander.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://exterior-everyone-gander.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()#校長是誰會用google搜尋正確答案
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [7]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於台灣新竹縣新豐鄉的私立科技大學。學校的創校理念源於《大學》「在明明德，在新民，在止於至善」的精義，旨在培養學生高尚品德、專業學問與優良技術，以達全人發展的境界，並以「堅毅、求新、創造」為校訓。

**歷史沿革**
明新科技大學的歷史可追溯至1966年3月1日創立的「明新工業專科學校」。初期設有機械、土木、工業管理三科。隨著時代發展，學校歷經多次改制與更名：
*   1993年，更名為「明新工商專科學校」。
*   1997年7月，奉教育部核准改制為「明新技術學院」。
*   2002年8月，再獲教育部核准升格為「明新科技大學」。
*   2018年12月，更名為「明新學校財團法人明新科技大學」。

**學院與特色**
目前，明新科技大學設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院以及共同教育學院等六個學院。學校積極配合國家經濟發展與產業需求，特別在半導體產業人才培育方面表現突出。明新科大被譽為產業最愛聘用的畢業生來源之一，尤其在半導體產業中名列前茅，是唯一入榜前五大的私立科大。

學校的育才特色包含「多元學習、全球視野、永續經營與技術創新」四大面向，引導學生進行跨領域學習，以培育符合產業需求的專業人才。明新科大也致力於推動產學合作，與半導體封測大廠及其他業界夥伴合作開辦專班，並設有「半導體產業設備廠務與檢測人才培育基地」，投入鉅資培育半導體設備開發、維修等實務人才。此外，明新科技大學也注重國際化發展，國際學生人數居全國之冠，並與多國學術單位簽訂合作計畫，推動雙聯學位與國際人才培育.

明新科技大學地處新竹縣新豐鄉，毗鄰新竹科學園區、新竹工業區及竹北生醫園區，享有豐富的產學合作資源與便利的交通。


In [12]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"U4bf2ca05145c0285eed4eb00c147e436","events":[{"type":"message","message":{"type":"text","id":"615035097855033465","quoteToken":"kk-XLClqZDEqzbwSSL7cVBUCyax4HhTKXOQCzFFh2fPJMun-Du94jT7mh3MsoM__K7PhCAgwR4cRs5YpmdvGHxHlIsXVZY0GP2uC0Hsl7DpFMVbIcvrnkIABUyzk9kLI69bUIgGhGS_17pK_oJYGVg","markAsReadToken":"jPkZi9O3bK7PE_PRd37XdR6RKDKOliFIij93DxnozaleUytET_HYoegS2NI7IfVeldFtBa6YoGEyeaCe2qKKE9VHgXL1bMaIEfBniFKIpgeqNpH0pOA-BNLfT__p0-Yl0UpAFyWRoGaV1XvstQks71SyKllktfXZ3ie-Oq9B1P8pbkKFGubMG7-AfvTGlNrLtu1RjJdPw2XCRxu5gFhwXA","text":"AI 簡介明新科大，30字以內"},"webhookEventId":"01KS6VWZ1TKDVDZB2EDHH3FJR9","deliveryContext":{"isRedelivery":false},"timestamp":1779420789354,"source":{"type":"user","userId":"U7de6d64df43af01c22a89a5ae823b57a"},"replyToken":"73600b802a0a4823bd6d2cb2b2558b0c","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:33:11] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U4bf2ca05145c0285eed4eb00c147e436","events":[{"type":"message","message":{"type":"text","id":"615035132214510148","quoteToken":"8RyKY2ZVD06IUw2Lnx1cRZgDUwvORBvZC7XhfRmvNvr00N8A3MneElN8_idK7gjooaT1e1t093bBeutNJ4_8tP-LNc3dioEMDPBPLgVBQhAfU5RTAfPSYCo_gTnEL6maRIp2FNCafjWkylz1z-HTgw","markAsReadToken":"9rgTT9kj8c7a6AP9TcJaUvkgXFNlJSgXVNvSS_fnLdWmOI3RpVxzu732RqFglXt77Jtw6sxoAQbJ2RKxRUX2oDZDdd88HyeGtILRCxoz2t34LRygZ7DtxS4kIq8Vz9TsVktTZyYW9MV6_MP6pEwTlwI5uuv1cJltCZxF3mBSVAEhvoxIU8UOfg_nPxtevRDizEkl9bGVRzpOAZsOLecWMQ","text":"AI 校長是誰"},"webhookEventId":"01KS6VXJNQ2EFJT42DGJQR2KFS","deliveryContext":{"isRedelivery":false},"timestamp":1779420809813,"source":{"type":"user","userId":"U7de6d64df43af01c22a89a5ae823b57a"},"replyToken":"62a5ad17bca84c828a0cefeb7afa7ffb","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 03:33:33] "POST / HTTP/1.1" 200 -
